# Qwen 7B LoRA 파인튜닝 — 결과와 남은 오답

공공 AI·IT RFP에서 먼저 검토할 조항을 좁히기 위한 실험입니다.
TF-IDF와 인코더 이후, 사전학습 지식과 라벨 지시를 활용하는 디코더 모델을 시험했습니다.

**실제로 완료한 학습은 Qwen2.5-7B-Instruct의 양자화 없는 LoRA입니다.**
대본은 QLoRA도 지원하지만, 아래 결과를 4bit QLoRA 성과로 표기하지 않습니다.

이 노트북은 저장된 예측·학습 로그를 읽고 점수와 오답 이동을 다시 계산합니다.
GPU, API 키, 네트워크, 모델 가중치 다운로드가 필요하지 않습니다.
입력 길이는 함께 보관한 토크나이저 감사 기록을 읽고, 현재 입력의 해시를 확인합니다.

흐름: 도입 이유 → 학습 방식 → 13문서 평가 → 경계 오답 이동 → 입력·판단 기준 진단.
관련 문서: [전체 프로젝트](22_project_summary.ipynb), [상세 진단](../reports/current/sllm_boundary_diagnosis.md), [포트폴리오](../docs/portfolio/rfp_portfolio_detailed.md).

In [ ]:
import os
os.environ['RFP_DATASET_VERSION'] = 'v5'
import hashlib
import json
import math
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'scripts').is_dir(), '저장소 루트 또는 notebooks 폴더에서 실행하세요.'
sys.path.insert(0, str(ROOT))
from scripts.evaluation.finetune_ensemble import load_members, vote, describe, macro, BOUNDARY

versions = ('v5', 'v7')
datasets, runs, members, golds, documents = {}, {}, {}, {}, {}
for version in versions:
    folder = ROOT / 'reports/current' / version
    datasets[version] = {r['requirement_uid']: r for r in map(json.loads,
        (ROOT / f'data/labels/label_dataset_{version}.jsonl').read_text(encoding='utf-8').splitlines())}
    records = [json.loads(line) for line in (folder / 'finetune_runs.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    selected = [r for r in records if r['config'].get('tag') == 'llm' and r['config']['fold'] == -1]
    assert len(selected) == 1
    runs[version] = selected[0]
    members[version], golds[version], documents[version] = load_members(folder / 'finetune_runs.jsonl', folder / 'model_candidate_oof.csv')
    predictions = [p for f in runs[version]['results'] for p in f['predictions']]
    assert len(predictions) == len({p['requirement_uid'] for p in predictions}) == 1345
    assert len(runs[version]['results']) == len(set(documents[version].values())) == 13
    assert all(p['gold'] == golds[version][p['requirement_uid']] == datasets[version][p['requirement_uid']]['primary_action'] for p in predictions)
    for fold in runs[version]['results']:
        ps = fold['predictions']
        assert math.isclose(macro([p['gold'] for p in ps], [p['pred'] for p in ps]), fold['test_macro_f1'], abs_tol=1e-12)
print('v5·v7 각각 13 fold / 평가 1,345건 / 정답 및 fold F1 검증 완료')

## 1. 왜 Qwen을 시험했나

인코더와 앙상블 이후에도 견적반영과 계약검토의 혼동이 남았습니다.
국방망 코드 어시스턴트처럼 공급·운영 조건을 해석해야 하는 사례가 동기였습니다.
사전학습 지식을 가진 7B 모델에 조치 정의를 제공하면 다른 오류를 보완할 수 있다고 기대했습니다.

다만 ‘문서 밖 상식의 효과’만 분리한 실험은 아닙니다.
모델 계열, system 지시, 학습 목표, 판정 방식이 함께 달라졌습니다.

## 2. 어떻게 학습하고 판정했나

1. 요구사항명과 본문의 앞 512토큰에 라벨 정의를 붙였습니다.
2. 사전학습 모델의 선형층에 LoRA를 적용했습니다. r=16, alpha=32, dropout=0.05입니다.
3. 프롬프트 토큰은 loss에서 제외하고 정답 라벨과 EOS 토큰만 학습했습니다. 클래스 가중치도 적용했습니다.
4. 세 후보 라벨과 EOS의 로그확률 합을 비교해 판정했습니다. 자유 생성 문자열을 파싱하지 않습니다.
5. fold별 검증 macro F1이 가장 높은 epoch의 어댑터 상태를 복원해 평가했습니다.

코드: [finetune_llm.py](../scripts/modeling/finetune_llm.py).
13문서 LODO를 사용하고 동결 앵커 100건은 평가에서 제외했습니다.
v5와 v7은 정답 라벨이 다르므로 각 버전 내부에서 모델을 비교합니다.

In [ ]:
keys = ['model', 'no_quant', 'lora_r', 'epochs', 'lr', 'batch_size', 'grad_accum', 'eval_batch_size', 'max_length', 'seed']
display(pd.DataFrame({v: {k: runs[v]['config'][k] for k in keys} for v in versions}))
assert all(runs[v]['config']['no_quant'] is True and not runs[v]['config']['smoke'] for v in versions)
assert runs['v5']['config']['system_prompt'] == runs['v7']['config']['system_prompt']
print('실제 실행: 양자화 없는 LoRA / 두 버전의 system 지시는 동일')

## 3. 학습은 진행됐나

아래 표는 13 fold의 epoch별 평균입니다. 검증값의 변화와 학습 loss를 함께 봅니다.
loss 하락만으로 일반화 성능이 좋아졌다고 판단하지 않습니다.
점수 비교에는 각 fold에서 선택한 모델의 별도 평가 결과를 사용합니다.

In [ ]:
history = pd.DataFrame([
    {'version': v, 'fold': f['fold_index'], **h}
    for v in versions for f in runs[v]['results'] for h in f['history']])
assert len(history) == 78 and history[['train_loss', 'validation_macro_f1']].notna().all().all()
display(history.groupby(['version', 'epoch'])[['train_loss', 'validation_macro_f1']].mean().round(4))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for v in versions:
    means = history[history.version == v].groupby('epoch').mean(numeric_only=True)
    axes[0].plot(means.index, means.train_loss, marker='o', label=v)
    axes[1].plot(means.index, means.validation_macro_f1, marker='o', label=v)
for ax, title in zip(axes, ['Mean training loss', 'Mean validation macro F1']):
    ax.set(title=title, xlabel='Epoch', xticks=[1, 2, 3]); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 4. 점수와 경계 혼동을 함께 비교합니다

주 지표는 문서별 macro F1의 평균입니다. 통합 OOF F1과 구분합니다.
경계 혼동은 견적반영과 계약검토를 서로 바꿔 예측한 건수입니다.
아래 값은 보고서의 숫자를 복사하지 않고 저장 예측에서 계산합니다.

In [ ]:
combos = {'TF-IDF': ('wc',), 'RoBERTa-large': ('ftL42',), 'Qwen LoRA': ('llm42',),
          '기존 앙상블': ('wc', 'ftB7', 'ftL42'), 'Qwen 앙상블': ('wc', 'ftL42', 'llm42')}
scores, predictions = [], {}
for v in versions:
    uids = list(golds[v])
    for name, combo in combos.items():
        pred = vote(members[v], combo, uids)
        predictions[v, name] = dict(zip(uids, pred))
        result = describe([golds[v][u] for u in uids], pred, [documents[v][u] for u in uids])
        scores.append({'version': v, 'model': name, **{k: result[k] for k in
            ['fold_mean_macro_f1', 'pooled_macro_f1', 'errors', 'boundary_errors']}})
scores = pd.DataFrame(scores).set_index(['version', 'model'])
display(scores.round(4))
delta = scores.loc[('v5', 'Qwen 앙상블'), 'fold_mean_macro_f1'] - scores.loc[('v5', '기존 앙상블'), 'fold_mean_macro_f1']
assert scores.loc[('v5', 'Qwen 앙상블'), 'boundary_errors'] == 111
assert scores.loc[('v7', 'Qwen LoRA'), 'boundary_errors'] == 110
print(f'v5 새 조합의 이득: {delta:+.4f}; 후속 실험 선별 기준 +0.016 미달')

v5 새 앙상블은 0.683으로 기존 0.671보다 약 0.011 높았습니다.
후속 실험 선별 기준 0.016에 못 미쳐 기존 주 구성을 유지했습니다.
이 기준은 이전 마스킹 변동에서 얻은 판단 기준이며, 이번 차이의 유의성 검정은 아닙니다.

v7 Qwen 단독은 0.665로 large의 0.640보다 높았습니다.
그러나 전체 오답은 둘 다 338건이고 경계 혼동은 96건에서 110건으로 늘었습니다.
한 seed 결과이므로 확정적인 모델 우위로 일반화하지 않습니다.

## 5. 경계 오답이 줄어든 경로는 무엇인가

경계 오답에서 빠졌어도 통상수용 오답으로 이동했다면 정답을 맞힌 것은 아닙니다.
새 조합이 고친 사례와 새로 틀린 사례를 나눠 계산합니다.

In [ ]:
movement = []
for v in versions:
    gold = golds[v]
    old, new = predictions[v, '기존 앙상블'], predictions[v, 'Qwen 앙상블']
    a = {u for u in gold if {gold[u], old[u]} == BOUNDARY}
    b = {u for u in gold if {gold[u], new[u]} == BOUNDARY}
    fixed = sum(new[u] == gold[u] for u in a - b)
    movement.append({'version': v, '기존 경계': len(a), '유지': len(a & b),
        '정답으로 수정': fixed, '다른 오답으로 이동': len(a - b) - fixed,
        '새 경계': len(b - a), '최종 경계': len(b),
        '최종 경계 중 Qwen 단독 정답': sum(members[v]['llm42'][u] == gold[u] for u in b)})
    assert len(a) - len(a - b) + len(b - a) == len(b)
display(pd.DataFrame(movement).set_index('version'))

v5에서 기존 경계 오답 20건을 맞혔지만 새 경계 오답도 17건 생겼습니다.
Qwen 단독이 맞힌 답이 다른 멤버의 표에 밀린 경우도 있습니다.
‘큰 모델도 같은 문제만 틀렸다’는 설명으로는 이 이동을 설명할 수 없습니다.

## 6. 긴 조항의 핵심 문장이 입력에서 빠졌습니다

보관된 토크나이저 감사 기록은 Qwen 7B의 실제 모델 ID로 계산했습니다.
정확한 revision과 입력 해시를 저장해, 다른 입력에 과거 길이 결과를 붙이지 않도록 했습니다.
아래 셀은 토큰화를 다시 수행하지 않습니다. 감사 기록 생성 시 전체 1,345건을 토큰화했습니다.
학습 로그에는 토크나이저 revision이 고정돼 있지 않아 해당 snapshot 기준의 재현입니다.

In [ ]:
audit = json.loads((ROOT / 'reports/current/sllm_input_audit.json').read_text(encoding='utf-8'))
texts = {u: datasets['v5'][u]['model_text'] for u in sorted(golds['v5'])}
assert hashlib.sha256(json.dumps(texts, ensure_ascii=False, sort_keys=True).encode()).hexdigest() == audit['model_text_sha256']
assert all(datasets['v7'][u]['model_text'] == t for u, t in texts.items())
assert set(audit['lengths']) == set(texts)
length_rows = []
for v in versions:
    for truncated in [False, True]:
        us = [u for u, n in audit['lengths'].items() if (n > audit['max_length']) == truncated]
        wrong = sum(members[v]['llm42'][u] != golds[v][u] for u in us)
        length_rows.append({'version': v, '512토큰 초과': truncated, '건수': len(us),
            'Qwen 오답': wrong, '오답률': wrong / len(us)})
display(pd.DataFrame(length_rows).round(4))
case_rows = []
for uid, case in audit['cases'].items():
    assert case['condition'] in texts[uid]
    assert (case['condition'] in case['visible']) == case['condition_visible']
    case_rows.append({'UID': uid, '전체 토큰': audit['lengths'][uid],
        '확인 조건': case['condition'], '512토큰 안에 보임': case['condition_visible']})
display(pd.DataFrame(case_rows).set_index('UID'))
print('토크나이저 revision:', audit['tokenizer_revision'])

- 석유공사 `QMR-001`: 987토큰 중 뒤쪽의 **정답률 90%** 조건이 빠졌습니다.
- 신용회복위원회 `PSR-002`: 1,499토큰 중 **최소 5년 부품·기술지원** 조건이 빠졌습니다.
- 신용회복위원회 `ECR-006`: 1,553토큰 중 **H200 도입** 조건이 빠졌습니다.
- 식약처 `PER-008`: **70토큰 전체를 읽어도** 성능 수치 조항을 견적반영으로 예측했습니다(v7).

긴 조항은 원래 어려울 수 있어 절단과 오답률의 관계를 인과효과로 해석하지 않습니다.
전체 입력을 주면 맞힌다는 결과도 아닙니다. 누락된 정보와 읽고도 놓친 규칙이 함께 확인됐습니다.

In [ ]:
# 핵심 조건과 입력 끝부분을 나란히 확인합니다.
for uid, case in audit['cases'].items():
    print(uid, '| 확인 조건:', case['condition'])
    print('모델에 제공된 조항의 끝:', case['visible'][-180:])
    print()

## 7. 라벨 기준과 실무 판단도 구분해야 합니다

보관 기간 협의(`SER-004`)와 표준랙 FMS 연동(`ECR-017`)은 v5에서 경계 오답이었지만,
v7에서는 라벨이 바뀌었고 Qwen이 새 정답을 맞혔습니다.
반면 플랫폼 통합관리(`SFR-012`)는 사용자의 견적반영 판단과 v7 정답표가 여전히 다릅니다.

이 세 사례만으로 정답표 전체의 품질을 추정하지 않습니다.
라벨러의 reasoning은 기존 판정 근거이며 독립적인 정답 증거가 아닙니다.

In [ ]:
uids = ['ccrs_ai_platform:SER-004', 'defense_intelligent_platform:ECR-017', 'ccrs_ai_platform:SFR-012']
display(pd.DataFrame([
    {'UID': uid, 'version': v, '동결 정답': golds[v][uid], 'Qwen': members[v]['llm42'][uid],
     '라벨러 근거': datasets[v][uid]['reasoning']}
    for uid in uids for v in versions]).set_index(['UID', 'version']))
human = json.loads((ROOT / 'reports/current/v5/boundary_review_user_judgments.json').read_text(encoding='utf-8'))
memo = next(r['memo'] for r in human['items'] if r['uid'] == 'ccrs_ai_platform:SFR-012')
print('기존 실무자 메모:', memo)

## 8. 포트폴리오에 남길 경험

> Qwen2.5-7B-Instruct를 LoRA로 파인튜닝했습니다.
> 라벨 토큰 학습과 후보 로그확률 판정을 구현했습니다.
> v5·v7 각각 13문서 LODO로 평가했습니다.
> 점수와 경계 오답을 함께 비교하고, 입력에서 빠진 판정 조건을 추적했습니다.

v7 단독 F1 0.665는 학습 결과로 기록할 수 있습니다.
v5 앙상블 0.683을 확정적인 성능 향상이나 배포 성과로 표현하지 않습니다.
입력 누락, 규칙 적용, 실무 판단 차이를 구분한 것이 후속 분석의 성과입니다.

추가 학습 계획은 [NEXT](../docs/NEXT.md)에서 관리합니다.
이 노트북은 학습을 시작하거나 동결 라벨을 수정하지 않습니다.